# Unit Test Generator - Fine-tuning with Unsloth

This notebook fine-tunes a Qwen2.5-Coder model for generating Python unit tests, saves it to GGUF format, and runs inference using llama-cpp-python.

## 1. Load Base Model

In [1]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto detection
load_in_4bit = True  # Use 4bit quantization to reduce memory

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/sachithra/miniconda3/envs/unsloth_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.5: Fast Qwen2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.628 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## 2. Add LoRA Adapters

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2025.12.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


## 3. Dataset Exploration
Let's first take a quick look at the structures of the training and testing splits in `KAKA22/CodeRM-UnitTest` before preparing the prompt templates.

In [3]:
from datasets import load_dataset

# Load raw datasets for exploration
dataset_train_explore = load_dataset("KAKA22/CodeRM-UnitTest", split="train")
dataset_test_explore = load_dataset("KAKA22/CodeRM-UnitTest", split="test")

print(f"Train dataset size: {len(dataset_train_explore)} rows")
print(f"Test dataset size: {len(dataset_test_explore)} rows")
print("-" * 50)
print(f"Train columns: {list(dataset_train_explore.features.keys())}")
print(f"Test columns:  {list(dataset_test_explore.features.keys())}")
print("-" * 50)

# Display a sample from the dataset
sample = dataset_train_explore[0]
print("Example 'question' from train set:")
print(sample['question'][:300] + "..." if len(sample['question']) > 300 else sample['question'])


Train dataset size: 17562 rows
Test dataset size: 62900 rows
--------------------------------------------------
Train columns: ['task_id', 'question', 'code_ground_truth', 'code_generate', 'unit_tests']
Test columns:  ['task_id', 'question', 'code_ground_truth', 'code_generate', 'unit_tests']
--------------------------------------------------
Example 'question' from train set:
There are $n$ candy boxes in front of Tania. The boxes are arranged in a row from left to right, numbered from $1$ to $n$. The $i$-th box contains $r_i$ candies, candies have the color $c_i$ (the color can take one of three values ​​— red, green, or blue). All candies inside a single box have the sa...


## 4. Prepare Training Data

In [4]:
# Defined the prompt format for Unit Testing
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    questions = examples["question"]
    codes     = examples["code_ground_truth"]
    unit_tests_lists = examples["unit_tests"]
    
    texts = []
    # Zip through the batch
    for question, code, tests_list in zip(questions, codes, unit_tests_lists):
        
        # safely extract the test suite
        if tests_list:
            if isinstance(tests_list, dict) and 'code' in tests_list:
                full_test_suite = "\n\n".join([str(c) for c in tests_list['code'] if c])
            elif isinstance(tests_list, list):
                full_test_suite = "\n\n".join([str(t['code']) for t in tests_list if isinstance(t, dict) and t.get('code')])
            elif isinstance(tests_list, str):
                full_test_suite = tests_list
            else:
                full_test_suite = ""
        else:
            full_test_suite = ""
            
        # We combine the Question and the Code into the Input field
        complex_input = f"Problem Description:\n{question}\n\nCode to Test:\n{code}"
        instruction = "Write a comprehensive Python unit test suite for the provided code."
        
        text = alpaca_prompt.format(instruction, complex_input, full_test_suite.strip()) + EOS_TOKEN
        texts.append(text)
        
    return { "text" : texts }

from datasets import load_dataset

# Load both the train and test splits of the dataset
train_dataset = load_dataset("KAKA22/CodeRM-UnitTest", split="train")
test_dataset = load_dataset("KAKA22/CodeRM-UnitTest", split="test")

#Shuffle and take a smaller subset of the test dataset for faster evaluation
test_dataset = test_dataset.shuffle(seed=42).select(range(1000))

# Apply the formatting function to both datasets
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
test_dataset = test_dataset.map(formatting_prompts_func, batched=True)

Map: 100%|██████████| 1000/1000 [00:01<00:00, 633.48 examples/s]


## 5. Train the Model

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import EarlyStoppingCallback

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = test_dataset, 
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1, # Set to 1 to reduce memory usage during tokenization
    args = SFTConfig(
        per_device_train_batch_size = 1, # Decreased to 1 to prevent CUDA out of memory
        per_device_eval_batch_size = 1,  # Decreased eval batch size
        dataset_num_proc = 1, 
        gradient_accumulation_steps = 8, # Increased to keep the effective batch size similar (was 2 * 4, now 1 * 8)
        warmup_steps = 5,
        max_steps = 100,          
        learning_rate = 2e-4,
        logging_steps = 10,
        eval_strategy = "steps",  
        eval_steps = 20,          
        save_strategy = "steps",  
        save_steps = 20,          
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss", 
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
    callbacks=[
        EarlyStoppingCallback(early_stopping_patience=2) 
    ]
)

[trl.trainer.sft_trainer|WARNING]You are using a per_device_train_batch_size of 1 with padding-free training. Using a batch size of 1 anihilate the benefits of padding-free training. Please consider increasing the batch size to at least 2.
Unsloth: Tokenizing ["text"] (num_proc=1): 100%|██████████| 1000/1000 [00:25<00:00, 39.79 examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [6]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 17,562 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 20,185,088 of 7,635,801,600 (0.26% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
20,0.403200,0.316079
40,0.342800,0.310856
60,0.378300,0.307658
80,0.339500,0.307796
100,0.346500,0.306958
120,0.390600,0.307557
140,0.350400,0.308496
160,0.331800,0.308547


Unsloth: Not an error, but Qwen2ForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


## 6. Model Evaluation (Generative Metrics)

For generative tasks like writing code, standard classification metrics like accuracy or F1-score don't work directly because there are many valid ways to structure code. Instead, we use NLP generative metrics:
* **BLEU:** Measures precision (how many n-grams in the generated test exist in the ground truth).
* **ROUGE:** Measures recall (structural overlap between generated code and ground truth).

*Note: The true gold standard for unit tests is Pass@k execution (running the code), but static text overlap metrics (BLEU/ROUGE) are significantly safer and standard for quick notebook benchmarking.*

In [9]:
# Install required evaluation libraries if missing:
# !pip install evaluate rouge_score
import re 
import evaluate
from tqdm import tqdm

# Load Hugging Face evaluation metrics
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")

# Prepare Unsloth model for 2x faster native inference
FastLanguageModel.for_inference(model)

predictions = []
references = []

# Evaluate on a small random subset (e.g., 100 samples) to save time in the notebook
eval_sample_size = 100
eval_subset = test_dataset.select(range(eval_sample_size))

print(f"Generating test cases for {eval_sample_size} samples...")
for i in tqdm(range(len(eval_subset))):
    sample = eval_subset[i]
    
    # 1. Format the prompt exactly like training
    complex_input = f"Problem Description:\n{sample['question']}\n\nCode to Test:\n{sample['code_ground_truth']}"
    instruction = "Write a comprehensive Python unit test suite for the provided code."
    prompt = alpaca_prompt.format(instruction, complex_input, "")
    
    # 2. Tokenize and Generate
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=512, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    try:
        response_text = decoded_output.split("### Response:")[1].strip()
        
        # Try to extract just the python code between markdown ticks
        code_blocks = re.findall(r'```(?:python)?\n(.*?)\n```', response_text, re.DOTALL)
        if code_blocks:
            response_text = "\n\n".join(code_blocks).strip()
            
    except IndexError:
        response_text = ""
        
    # 4. Extract ground truth safely
    try:
        ut = sample['unit_tests']
        if isinstance(ut, dict) and 'code' in ut:
            ground_truth = "\n\n".join([str(c) for c in ut['code'] if c]).strip()
        elif isinstance(ut, list):
            ground_truth = "\n\n".join([str(t['code']) for t in ut if isinstance(t, dict) and t.get('code')]).strip()
        elif isinstance(ut, str):
            ground_truth = ut.strip()
        else:
            ground_truth = ""
    except Exception as e:
        print(f"Failed to parse ground truth for a sample: {e}")
        ground_truth = ""
        
    predictions.append(response_text)
    references.append([ground_truth]) # BLEU expects a list of references

# Calculate Scores
print("\n--- Evaluation Results ---")

filtered_predictions = []
filtered_references = []
empty_preds_count = 0

for pred, ref in zip(predictions, references):
    # Only keep samples where a valid ground truth exists
    if not ref[0].strip():
        continue
        
    # If the model generated absolutely nothing, insert a dummy token
    # so the BLEU metric length division doesn't hit 0 (this will safely score 0.0)
    if not pred.strip():
        empty_preds_count += 1
        pred = "# empty" 
        
    filtered_predictions.append(pred)
    filtered_references.append(ref)

if empty_preds_count > 0:
    print(f"⚠️ Warning: The model generated empty responses for {empty_preds_count} out of {len(filtered_references)} valid samples.")

if len(filtered_references) == 0:
    print("Warning: All ground truth references were empty! Cannot compute BLEU/ROUGE.")
else:
    rouge_results = rouge_metric.compute(
        predictions=filtered_predictions, 
        references=[ref[0] for ref in filtered_references]
    )
    bleu_results = bleu_metric.compute(
        predictions=filtered_predictions, 
        references=filtered_references
    )

    print(f"BLEU Score:   {bleu_results['bleu'] * 100:.2f}%")
    print(f"ROUGE-1:      {rouge_results['rouge1'] * 100:.2f}%")
    print(f"ROUGE-2:      {rouge_results['rouge2'] * 100:.2f}%")
    print(f"ROUGE-L:      {rouge_results['rougeL'] * 100:.2f}%")

Generating test cases for 100 samples...


100%|██████████| 100/100 [33:52<00:00, 20.32s/it]



--- Evaluation Results ---
BLEU Score:   0.00%
ROUGE-1:      5.56%
ROUGE-2:      4.73%
ROUGE-L:      4.96%


## 7. Save Merged Model & Convert to GGUF

In [8]:
# Save model in standard HF format, merged with LoRA in 16-bit
output_dir = "merged_model_16bit"

model.save_pretrained_merged(output_dir, tokenizer, save_method="merged_16bit")
print(f"✅ Saved merged 16-bit Hugging Face model to '{output_dir}/'")

Found HuggingFace hub cache directory: /home/sachithra/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [1:10:34<00:00, 1058.74s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [00:58<00:00, 14.74s/it]


Unsloth: Merge process complete. Saved to `/home/sachithra/Desktop/FYP/merged_model_16bit`
✅ Saved merged 16-bit Hugging Face model to 'merged_model_16bit/'


### Manual GGUF Conversion

Since Unsloth's built-in GGUF saving tries to recompile `llama.cpp` and currently fails, you will convert the created `merged_model_16bit` folder manually. 

Once the above cell finishes saving the model, open your terminal, ensure you are in the `FYP` directory, and run the standard conversion script provided inside your cloned `llama.cpp` repo:


# Convert your 16-bit merged model directly to a 4-bit GGUF
```bash
python llama.cpp/convert_hf_to_gguf.py ./merged_model_16bit --outfile gguf_model/unsloth.q4_k_m.gguf --outtype q4_k_m
```
```bash
python llama.cpp/convert_hf_to_gguf.py ./merged_model_16bit --outfile gguf_model/unsloth.Q8_0.gguf --outtype q8_0
```

# Command for building

git clone https://github.com/ggerganov/llama.cpp

cd llama.cpp

cmake -B build

cmake --build build --config Release